In [1]:
# Install (run once)
!pip install pandas numpy scikit-learn mlxtend gradio

  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.22-py3-none-any.whl.metadata (1.8 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.0.0-py3-no

In [2]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import apriori, association_rules
import gradio as gr

e:\Ecommerce Recommendation System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load Data
orders = pd.read_csv("../data/orders.csv")
order_products = pd.read_csv("../data/order_products__prior.csv")
products = pd.read_csv("../data/products.csv")

df = order_products.merge(orders, on="order_id")
df = df.merge(products, on="product_id")
df = df[['user_id', 'order_id', 'product_id', 'product_name']]

In [4]:
# Preprocessing (OPTIMIZED)
df = df.sample(n=200000, random_state=42)

top_products = df['product_name'].value_counts().head(100).index
df = df[df['product_name'].isin(top_products)]


In [5]:
# Keep only orders with multiple items
order_counts = df.groupby('order_id')['product_id'].count()
valid_orders = order_counts[order_counts > 1].index
df = df[df['order_id'].isin(valid_orders)]

In [6]:
# Train-test split
train, test = train_test_split(df, test_size=0.2, random_state=42)

# Basket creation
basket = train.groupby(['order_id', 'product_name'])['product_id'] \
              .count().unstack()

basket = basket.notnull()

In [7]:
# Apriori
frequent_items = apriori(basket, min_support=0.003, use_colnames=True)
print("Frequent itemsets:", len(frequent_items))

rules = association_rules(frequent_items, metric="confidence", min_threshold=0.05)
print("Rules generated:", len(rules))

Frequent itemsets: 126
Rules generated: 50


In [12]:
def recommend(product_name):
    product_name = product_name.lower()
    recommendations = []

    for _, row in rules.iterrows():
        antecedents = [str(i).lower() for i in row['antecedents']]
        consequents = list(row['consequents'])
        confidence = row['confidence']

        if any(product_name in item for item in antecedents):
            for item in consequents:
                recommendations.append((item, confidence))

    # Sort by confidence
    recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)

    # Remove duplicates
    seen = set()
    final = []
    for item, conf in recommendations:
        if item not in seen:
            final.append(f"{item} (score: {round(conf,2)})")
            seen.add(item)

    # Fallback
    if not final:
        top = train['product_name'].value_counts().head(5).index.tolist()
        return ["No strong rules found"] + top

    return final[:5]

In [14]:
# Gradio UI
def gradio_recommend(product):
    result = recommend(product)
    return "\n".join(result)

interface = gr.Interface(
    fn=gradio_recommend,
    inputs=gr.Textbox(label="Enter Product Name"),
    outputs=gr.Textbox(label="Recommendations"),
    title="🛒 E-commerce Recommendation System",
    description="Enter a product name to get recommendations"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


e:\Ecommerce Recommendation System\.venv\Lib\site-packages\gradio\routes.py:1402: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Python313\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
C:\Python313\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
e:\Ecommerce Recommendation System\.venv\Lib\site-packages\gradio\queueing.py:168: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  .infer_objects(copy=False)  # type: ignore
C:\Python313\Lib\contextlib.py:148: Pandas4Warning: 'future.no

Created dataset file at: .gradio\flagged\dataset1.csv
